# Phase 2: Baseline Model Building

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')

### Step 1: Prepare Data for Modeling

In [ ]:
# Assuming 'train' is the cleaned dataframe from Phase 1
target_col = 'Class/ASD'
X = train.drop(columns=[target_col])
y = train[target_col]

# Calculate class weights to handle imbalance (Class 0: 77%, Class 1: 23%)
scale_weight = len(y[y == 0]) / len(y[y == 1])
print(f"Scale Positive Weight for XGBoost: {scale_weight:.2f}")

### Step 2: Define Models and Cross-Validation Strategy

In [ ]:
# Define 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define scoring metrics
scoring_metrics = ['accuracy', 'f1', 'roc_auc']

# Initialize Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, scale_pos_weight=scale_weight),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42) # Added based on research paper
}

### Step 3: Train and Evaluate Models

In [ ]:
results_list = []
print("Evaluating Models using 5-Fold Cross-Validation...\n")

for name, model in models.items():
    print(f"Training {name}...")
    cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring_metrics)
    results_list.append({
        'Model': name,
        'Accuracy': cv_results['test_accuracy'].mean(),
        'F1-Score': cv_results['test_f1'].mean(),
        'ROC-AUC': cv_results['test_roc_auc'].mean()
    })

results_df = pd.DataFrame(results_list).round(4)

print("\n--- LEADERBOARD ---")
display(results_df.sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True))